# Surrogates (CatBoost)

- Um modelo por objetivo (2 ou 3), features = todas as `x_*`.
- Regiões: grade 4×4 sempre em `(x_1, x_2)`.
- Entrada: `data/dataframes/{nome}/df_{nome}.parquet` (notebook 1).
- Saída: `data/dataframes/{nome}/df_{nome}_previsao.parquet`.
- Visualizações: `display_pareto_fronts_catalog` + `display_decision_space` (com `cmap=plasma`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

from src import problems as _problems_mod
from src.processing import (
    treinar_catboost,
    prever_catboost,
    find_pareto_front,
    find_pareto_front_nd,
    injetar_ruidos_parametrizados,
    adicionar_regioes_bins,
)
from src.visualization import (
    plot_grid_16_ruidos,
    display_pareto_fronts_catalog,
    display_decision_space,
)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['lines.linewidth'] = 0.5

PROBLEM_NAMES = [
    'MMF1', 'MMF4', 'MMF11_L', 'MMF16_L3', 'MMF16_20',
    'ZDT1', 'ZDT3', 'ZDT4', 'ZDT6',
    'DTLZ1', 'DTLZ2', 'DTLZ3', 'DTLZ4', 'DTLZ7',
    'WFG1', 'WFG2', 'WFG4', 'WFG5', 'WFG9',
]

THREE_OBJ = {
    'MMF16_L3', 'MMF16_20', 'DTLZ1', 'DTLZ2', 'DTLZ3', 'DTLZ4', 'DTLZ7',
}


def _instantiate_problem(name):
    return getattr(_problems_mod, name)()


_NDS = NonDominatedSorting(method="efficient_non_dominated_sort")


def _extract_pareto_set(df, x_cols, obj_cols, minimize=True):
    """Return X rows belonging to the first non-dominated front.

    Uses Efficient Non-dominated Sort (ENS) with early stop at the first
    front: O(N * log^(M-1) N) time, no NxN matrix, viable for N >> 1e5.
    """
    F = df[list(obj_cols)].values.astype(float)
    if not minimize:
        F = -F
    idx = _NDS.do(F, only_non_dominated_front=True)
    return df.iloc[idx][list(x_cols)].values


def analise_visual_surrogate(name, problem, df_noisy, df_previsao, df_validacao):
    """
    Visualisation using display_pareto_fronts_catalog + display_decision_space.
    All three empirical fronts are mapped to **f_original** (clean) space so
    they are directly comparable to the true PF.
    """
    n_obj = problem.n_obj
    x_cols = [f'x_{k}' for k in range(1, problem.n_var + 1)]
    f_orig = [f'f{j}_original' for j in range(1, n_obj + 1)]
    f_noise = [f'f{j}' for j in range(1, n_obj + 1)]
    f_pred = [f'f{j}_predicted' for j in range(1, n_obj + 1)]

    # --- Pareto fronts (identified in their own spaces) ---
    if n_obj == 2:
        pf_orig = find_pareto_front(df_noisy, minimize=True,
                    fitness1='f1_original', fitness2='f2_original')
        pf_noise = find_pareto_front(df_noisy, minimize=True,
                    fitness1='f1', fitness2='f2')
        pf_pred  = find_pareto_front(df_previsao, minimize=True,
                    fitness1='f1_predicted', fitness2='f2_predicted')
    else:
        pf_orig  = find_pareto_front_nd(df_noisy, f_orig, minimize=True)
        pf_noise = find_pareto_front_nd(df_noisy, f_noise, minimize=True)
        pf_pred  = find_pareto_front_nd(df_previsao, f_pred, minimize=True)

    # F values always in *original* (clean) space for visual comparison
    F_orig  = pf_orig[f_orig].values
    F_noise = pf_noise[f_orig].values
    F_pred  = pf_pred[f_orig].values

    # PS in decision space
    X_ps_orig  = pf_orig[x_cols].values
    X_ps_noise = pf_noise[x_cols].values
    X_ps_pred  = pf_pred[x_cols].values

    # Background landscape (original values)
    F_landscape = df_noisy[f_orig].values
    X_landscape = df_noisy[x_cols].values

    # ── Objective space ──
    print('\n' + '=' * 80)
    print('ESPAÇO DE OBJETIVOS')
    print('=' * 80)

    display_pareto_fronts_catalog(
        problem,
        F_landscape,
        pareto_fronts_list=[F_orig, F_noise, F_pred],
        front_names=[
            'PF ORIGINAL (sem ruído)',
            'PF RUIDOSO (f original)',
            'PF PREVISTO (f original)',
        ],
        front_colors=['red', 'orange', 'dodgerblue'],
        sample_size=100_000,
        title=f'{name} — Espaço de Objetivos',
    )

    # ── Decision space (with multi-variant heatmaps + WAPE for 2-var) ──
    print('\n' + '=' * 80)
    print('ESPAÇO DE DECISÕES')
    print('=' * 80)

    X_true_ps = problem.true_pareto_front()[0]

    F_variants = None
    df_wape = None
    if problem.n_var == 2:
        F_variants = [
            ('Original (sem ruído)', df_previsao[f_orig].values),
            ('Ruidoso (com ruído)',   df_previsao[f_noise].values),
            ('Previsto (surrogate)',  df_previsao[f_pred].values),
        ]
        df_wape = df_previsao
        X_landscape_show = df_previsao[x_cols].values
        F_landscape_show = df_previsao[f_orig].values
    else:
        X_landscape_show = X_landscape
        F_landscape_show = F_landscape

    display_decision_space(
        problem,
        X_landscape_show,
        F_landscape_show,
        X_true=X_true_ps,
        X_emp_list=[X_ps_orig, X_ps_noise, X_ps_pred],
        emp_names=['PS ORIGINAL', 'PS RUIDOSO', 'PS PREVISTO'],
        emp_colors=['red', 'orange', 'dodgerblue'],
        cmap='plasma',
        title=name,
        F_variants=F_variants,
        df_wape=df_wape,
    )


def train_surrogate_pipeline(
    name,
    noise_config,
    sampling_method='sobol',
    obs_treino=500,
    n_bins_x1=4,
    n_bins_x2=4,
    cat_verbose=0,
    plot=True,
):
    """Carrega parquet do notebook 1, regiões 4x4 em (x_1,x_2), ruído, CatBoost, salva previsão."""
    n_regioes = n_bins_x1 * n_bins_x2
    base = Path('data/dataframes') / name
    if sampling_method not in ['sobol', 'latin_hypercube', 'random']:
        raise ValueError(f"sampling_method must be one of ['sobol', 'latin_hypercube', 'random'], got '{sampling_method}'")
    path_in = base / f'df_{name}_landscape_{sampling_method}.parquet'
    df = pd.read_parquet(path_in)
    df = adicionar_regioes_bins(df, n_bins_x1=n_bins_x1, n_bins_x2=n_bins_x2)

    problem = _instantiate_problem(name)
    n_obj = problem.n_obj
    x_cols = [f'x_{k}' for k in range(1, problem.n_var + 1)]

    df['f1_original'] = df['f1'].copy()
    df['f2_original'] = df['f2'].copy()
    if n_obj >= 3:
        df['f3_original'] = df['f3'].copy()

    df_noisy = injetar_ruidos_parametrizados(df, noise_config, col_regiao='regiao')

    df_treino = df_noisy.groupby('regiao', group_keys=False).apply(
        lambda x: x.sample(n=int(obs_treino / n_regioes), random_state=42))
    df_valid = df_noisy.loc[~df_noisy.index.isin(df_treino.index)].copy()

    print(f'Treino: {df_treino.shape}, Validação: {df_valid.shape}')

    models = {}
    for j in range(1, n_obj + 1):
        fj = f'f{j}'
        models[fj] = treinar_catboost(
            df_treino,
            features=x_cols,
            target=fj,
            colunas_categoricas=[],
            loss_function='MAE',
            verbose=cat_verbose,
        )

    df_pre = df_valid.copy()
    for j in range(1, n_obj + 1):
        fj = f'f{j}'
        df_pre = prever_catboost(models[fj], df_pre, x_cols, col_name=f'{fj}_predicted')

    for j in range(1, n_obj + 1):
        fj = f'f{j}'
        mae = (df_pre[f'{fj}_predicted'] - df_pre[fj]).abs().mean()
        den = df_pre[fj].replace(0, np.nan).mean()
        wape = mae / den if den else np.nan
        print(f'{name} {fj} MAE={mae:.6g} WAPE={wape:.6g}')

    out_path = base / f'df_{name}_previsao.parquet'
    df_pre.to_parquet(out_path)
    print(f'Salvo em {out_path}')

    if plot:
        analise_visual_surrogate(name, problem, df_noisy, df_pre, df_valid)

    return {'df_noisy': df_noisy, 'df_previsao': df_pre,
            'df_validacao': df_valid, 'models': models}


noise_config = {
    1:  {'dist': 'bimodal',     'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'p': [0.6, 0.4], 'mu1': -2, 'sd1': 0.5, 'mu2': 2, 'sd2': 1}},
    2:  {'dist': 'student_t',   'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'df': 3}},
    3:  {'dist': 'lognormal',   'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'mean': 0, 'sigma': 0.8}},
    4:  {'dist': 'poisson',     'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'lam': 5.0}},
    5:  {'dist': 'binomial',    'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'n': 10, 'p': 0.5}},
    6:  {'dist': 'geometric',   'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'p': 0.3}},
    7:  {'dist': 'exponential', 'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'scale': 1.0}},
    8:  {'dist': 'uniform',     'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'low': -1.0, 'high': 1.0}},
    9:  {'dist': 'laplace',     'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'loc': 0.0, 'scale': 1.0}},
    10: {'dist': 'gamma',       'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'shape': 2.0, 'scale': 2.0}},
    11: {'dist': 'beta',        'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'a': 0.5, 'b': 0.5}},
    12: {'dist': 'rayleigh',    'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'scale': 1.0}},
    13: {'dist': 'weibull',     'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'a': 1.5}},
    14: {'dist': 'logistic',    'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'loc': 0.0, 'scale': 1.0}},
    15: {'dist': 'pareto',      'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {'a': 3.0}},
    16: {'dist': 'rademacher',  'target_mean': 0.0,  'forca_ruido': 0.2,  'params': {}},
}

plot_grid_16_ruidos(noise_config)


## MMF1

In [ ]:
train_surrogate_pipeline('MMF1', noise_config)

## MMF4

In [ ]:
train_surrogate_pipeline('MMF4', noise_config)

## MMF11_L

In [ ]:
train_surrogate_pipeline('MMF11_L', noise_config)

## MMF16_L3

In [ ]:
train_surrogate_pipeline('MMF16_L3', noise_config)

## MMF16_20

In [ ]:
train_surrogate_pipeline('MMF16_20', noise_config)

## ZDT1

In [ ]:
train_surrogate_pipeline('ZDT1', noise_config)

## ZDT3

In [ ]:
train_surrogate_pipeline('ZDT3', noise_config)

## ZDT4

In [ ]:
train_surrogate_pipeline('ZDT4', noise_config)

## ZDT6

In [ ]:
train_surrogate_pipeline('ZDT6', noise_config)

## DTLZ1

In [ ]:
train_surrogate_pipeline('DTLZ1', noise_config)

## DTLZ2

In [ ]:
train_surrogate_pipeline('DTLZ2', noise_config)

## DTLZ3

In [ ]:
train_surrogate_pipeline('DTLZ3', noise_config)

## DTLZ4

In [ ]:
train_surrogate_pipeline('DTLZ4', noise_config)

## DTLZ7

In [ ]:
train_surrogate_pipeline('DTLZ7', noise_config)

## WFG1

In [ ]:
train_surrogate_pipeline('WFG1', noise_config)

## WFG2

In [ ]:
train_surrogate_pipeline('WFG2', noise_config)

## WFG4

In [ ]:
train_surrogate_pipeline('WFG4', noise_config)

## WFG5

In [ ]:
train_surrogate_pipeline('WFG5', noise_config)

## WFG9

In [ ]:
train_surrogate_pipeline('WFG9', noise_config)